# IE643 KD diagnostic (CPU, ~3-5 min, no GPU quota)
Ablation 1 showed every token-KD run far below CE-only (paired CE14-F1 −0.23 to −0.29), with decodes full of the teacher's
formatting (`**`, `[1]`, `[cardiomegaly]`). This notebook measures, from the cache alone, what the KD targets contain:
how often the teacher's top-1 is the ground-truth token, where it disagrees (first token, sentence starts, elsewhere), whether
disagreements are case/format variants of the GT token, how much target mass sits on formatting tokens at tau 1 and 2,
and how much probability the top-20 covers.

**Inputs:** `IE643 Cleaning Step 7` (reports_clean.csv) and the teacher-cache merge version (`teacher_final/`). Accelerator None. Internet On (tokenizer).

In [ ]:
OFFLINE_TEST = False
import os, re, glob, json, collections
import numpy as np, pandas as pd
INPUT_ROOT = "/kaggle/input"
csv = [p for p in glob.glob(f"{INPUT_ROOT}/**/reports_clean.csv", recursive=True)
       if "impression_final" in pd.read_csv(p, nrows=0).columns]
tdir = sorted({os.path.dirname(p) for p in glob.glob(f"{INPUT_ROOT}/**/teacher_final/teacher_topk.npz", recursive=True)})
assert len(csv) == 1 and len(tdir) == 1, (csv, tdir)
df = pd.read_csv(csv[0], low_memory=False)
df["impression_final"] = df["impression_final"].fillna("").astype(str)
df["use_for_training"] = df["use_for_training"].astype(str).str.lower().isin(["true", "1"])
TK = np.load(f"{tdir[0]}/teacher_topk.npz")
off, IDS, LP = TK["offsets"], TK["ids"], TK["logp"].astype(np.float32)
row = {int(v): k for k, v in enumerate(TK["idx"])}
TC = {r["idx"]: r for r in map(json.loads, open(f"{tdir[0]}/teacher_cache.jsonl"))}
V = 50257
if OFFLINE_TEST:
    import zlib
    enc = lambda t: [zlib.crc32(w.encode()) % 50000 for w in t.split()]
    dec = lambda i: f"w{i}"
else:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("distilgpt2")
    enc = lambda t: tok(t)["input_ids"]
    dec = lambda i: tok.decode([int(i)]) if i < V else f"<teacher-only {i}>"
print("csv", csv[0], "| teacher", tdir[0], "| K =", IDS.shape[1], "| rows", len(IDS))
print("first train impressions as cached (case check):", df[df.subset == "train"].impression_final.head(3).tolist())

In [ ]:
# per-token records over the train split: position class, teacher top-1, GT rank, mass
tr = df[(df.subset == "train") & df.use_for_training]
STYLE = re.compile(r"[\*\[\]#\n]|^\s*\d+\.?\s*$")            # markdown, list brackets, newlines, bare list numbers
recs, mis = [], 0
for i, t in zip(tr["idx"].astype(int), tr.impression_final):
    r = row.get(i)
    if r is None or not TC.get(i, {}).get("topk_ok", False):
        continue
    g = enc(t)
    a, b = int(off[r]), int(off[r + 1])
    if b - a != len(g):
        mis += 1; continue
    prev = ""
    for p in range(len(g)):
        ids, lp = IDS[a + p], LP[a + p]
        cls = "first token" if p == 0 else ("sentence start" if prev.rstrip().endswith(".") else "inside sentence")
        recs.append((i, p, cls, g[p], ids, lp))
        prev = dec(g[p])
print(f"train studies with aligned rows: {len(set(x[0] for x in recs))} | misaligned {mis} | tokens {len(recs)}")

In [ ]:
def target(lp, ids, tau):
    """the renormalised top-k target the student sees (student-vocab ids only), as in losses.topk_kd_loss"""
    valid = ids < V
    z = np.where(valid, lp / tau, -1e9)
    z = np.exp(z - z.max()); z = z * valid
    return z / max(z.sum(), 1e-12)

style_id = {}
def is_style(i):
    if i not in style_id:
        s = dec(i)
        style_id[i] = bool(STYLE.search(s)) or i >= V
    return style_id[i]

def is_case_variant(i, gt):
    a, b = dec(i), dec(gt)
    return i != gt and a.strip().lower() == b.strip().lower() and a.strip() != ""

rows = []
for (i, p, cls, gt, ids, lp) in recs:
    pr = np.exp(lp)
    t1, t2 = target(lp, ids, 1.0), target(lp, ids, 2.0)
    st = np.array([is_style(int(x)) for x in ids])
    gt_in = ids == gt
    rows.append({"cls": cls, "top1_is_gt": int(ids[np.argmax(lp)] == gt),
                 "gt_in_topk": int(gt_in.any()), "p_gt": float(pr[gt_in].sum()),
                 "topk_mass": float(pr.sum()), "teacher_only_mass": float(pr[ids >= V].sum()),
                 "top1_style": int(st[np.argmax(lp)]), "top1_case_variant": int(is_case_variant(int(ids[np.argmax(lp)]), gt)),
                 "target_style_mass_tau1": float(t1[st].sum()), "target_style_mass_tau2": float(t2[st].sum()),
                 "target_gt_mass_tau1": float(t1[gt_in].sum()), "target_gt_mass_tau2": float(t2[gt_in].sum()),
                 "top1": int(ids[np.argmax(lp)]), "gt": int(gt)})
R = pd.DataFrame(rows)
cols = ["top1_is_gt", "gt_in_topk", "p_gt", "topk_mass", "teacher_only_mass", "top1_style", "top1_case_variant",
        "target_gt_mass_tau1", "target_gt_mass_tau2", "target_style_mass_tau1", "target_style_mass_tau2"]
summ = pd.concat([R[cols].mean().rename("all tokens"), R.groupby("cls")[cols].mean().T], axis=1)
print("token counts:", R.cls.value_counts().to_dict())
print(summ.round(4).to_string())

In [ ]:
# what the teacher prefers when its top-1 is not the GT token
wrong = R[R.top1_is_gt == 0]
for cls in ["first token", "sentence start", "inside sentence"]:
    w = wrong[wrong.cls == cls]
    c = collections.Counter(w["top1"]).most_common(15)
    print(f"\n{cls}: teacher top-1 != GT on {len(w)}/{(R.cls == cls).sum()} tokens; most common teacher top-1:")
    print("   " + " | ".join(f"{dec(k)!r} {v}" for k, v in c))
    g = collections.Counter(w["gt"]).most_common(8)
    print("   GT tokens there: " + " | ".join(f"{dec(k)!r} {v}" for k, v in g))
print("\ncached per-study mean teacher log p(GT):", np.nanmean([float(r["teacher_gt_logp_mean"]) for r in TC.values() if r.get("split") == "train" and r.get("teacher_gt_logp_mean") is not None]))
json.dump({"summary": summ.round(5).to_dict()}, open("/kaggle/working/kd_diagnostic.json" if not OFFLINE_TEST else "kd_diag.json", "w"), indent=1)
print("\nKD DIAGNOSTIC DONE")